In [0]:
import requests
import base64
import json
from datetime import datetime
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, md5
from urllib.parse import urlparse
import re
import time
from datetime import datetime, timedelta
from pyspark.sql.functions import from_json, col

In [0]:
def run_qc(qc_id,job_name,run_id):

    qc_config = get_qc_config(qc_id)

    qc_name = qc_config["qc_name"]
    threshold_value = qc_config.get("threshold_expected_value")
    expression = qc_config.get("expression")

    src_query = qc_config.get("source_query")
    tgt_query = qc_config.get("target_query")

    src_path = qc_config.get("source_path")
    tgt_path = qc_config.get("target_path")

    src_type = qc_config.get("source_type")
    tgt_type = qc_config.get("target_type")
    store_error_records=qc_config.get("store_error_records")
    err_limit=qc_config.get("err_limit")
    error_query=qc_config.get("error_query")

    src_type = src_type.lower() if src_type else None
    tgt_type = tgt_type.lower() if tgt_type else None

    # QC EXECUTION
    if qc_name == "count_check":

        status, msg = count_check(
            src_type=src_type,
            tgt_type=tgt_type,
            src_query=src_query,
            tgt_query=tgt_query,
            threshold=threshold_value,
            expression=expression,
            src_path=src_path,
            tgt_path=tgt_path,
            qc_name=qc_name
        )

    elif qc_name == "manual_check":

        status, msg = manual_check(
             src_type=src_type,
            tgt_type=tgt_type,
            src_query=src_query,
            tgt_query=tgt_query,
            threshold=threshold_value,
            expression=expression,
            src_path=src_path,
            tgt_path=tgt_path,
            qc_name=qc_name
        )
        
    # Duplicate check
    elif qc_name == "row_hash_duplicate_check":

        status, msg = row_hash_duplicate_check(
            src_type=src_type,
            tgt_type=tgt_type,
            src_query=src_query,
            tgt_query=tgt_query,
            threshold=threshold_value
        )
    

    elif qc_name == "column_exist_check":

        results = columns_check(
            target_columns=qc_config.get("target_columns"),
            qc_name=qc_name,
            threshold=threshold_value,
            src_query=src_query,
            tgt_query=tgt_query
        )

        status = "PASSED"
        messages = []

        for r in results.values():
            messages.append(r["msg"])
            if r["status"] == "FAILED":
                status = "FAILED"

        msg = " | ".join(messages)



    else:
        raise Exception(f"Unsupported QC name: {qc_name}")

    # LOG & DISPLAY
    write_log(qc_id, qc_name, msg, status, src_path, tgt_path,job_name,run_id)
    display_status(qc_id, qc_name, src_path, tgt_path, msg, status,src_type,tgt_type,store_error_records,err_limit,error_query,job_name,run_id,tgt_query)


In [0]:
# Load QC configuration row from qc_config table
def get_qc_config(qc_id):
    config_path = dbutils.widgets.get("config_table_path")

    df = spark.sql(f"""
        SELECT *
        FROM {config_path}
        WHERE qc_id = {qc_id} AND active_flag = 1
    """).collect()

    if not df:
        raise Exception(f"No ACTIVE QC rule found for qc_id = {qc_id}")

    return df[0].asDict()



# Execute SQL and return single value
def run_query(query, qc_name):
    validate_query_text(query, qc_name)
    df = spark.sql(query)              
    value = validate_single_value(df, qc_name)  
    return value



def run_query_with_date(query, qc_name, table_name):
    validate_query_text(query, qc_name)

    # detect date column (CreateDate or create_date)
    date_col = get_date_column(table_name)
    if not date_col:
        raise Exception(f"No create date column found for table {table_name}")

    # get widget values (expected format: YYYY-MM-DD)
    start_dt = dbutils.widgets.get("start_dt")
    end_dt = dbutils.widgets.get("end_dt")

    if not start_dt or not end_dt:
        raise Exception("start_dt and end_dt widgets must be provided")

    # build query
    if "where" in query.lower():
        final_query = f"""
        {query}
        AND TO_DATE({date_col}) BETWEEN DATE('{start_dt}') AND DATE('{end_dt}')
        """
    else:
        final_query = f"""
        {query}
        WHERE TO_DATE({date_col}) BETWEEN DATE('{start_dt}') AND DATE('{end_dt}')
        """

    df = spark.sql(final_query)
    return validate_single_value(df, qc_name)




def get_date_column(table_name):
    cols = [c.lower() for c in spark.table(table_name).columns]

    if "createdate" in cols:
        return "CreateDate"
    elif "create_date" in cols:
        return "create_date"
    else:
        return None
 


# Display QC result
def display_status(qc_id,qc_name, src_path,tgt_path, msg, status,src_type,tgt_type,store_error_records,err_limit,error_query,job_name,run_id,tgt_query):
    print("\n================ QC RESULT ================")
    print(f"QC ID       : {qc_id}")
    print(f"Source      : {src_path}")
    print(f"Target      : {tgt_path}")
    print(f"Message     : {msg}")
    print(f"Final Result: {status}")
    print("===========================================\n")

    if src_type=="api" and tgt_type=="table":
        tgt_count=run_query(tgt_query, qc_name)
        if str(status).strip().upper() == "FAILED" and tgt_count == 0:
            dbutils.jobs.taskValues.set(key="qc_result", value="PASSED")

        elif str(status).strip().upper() == "FAILED":
            dbutils.jobs.taskValues.set(key="qc_result", value="FAILED")
            
        else:
            dbutils.jobs.taskValues.set(key="qc_result", value="PASSED")
            
    else:
        if str(status).strip().upper() == "FAILED":
            if store_error_records:
                insert_qc_error_records_sql(
                    qc_id=qc_id,
                    job_name=job_name,
                    run_id=run_id,
                    store_error_records=store_error_records,
                    error_query=error_query,
                    err_limit=err_limit
                )

            
            
            raise Exception(f"QC FAILED for QC ID {qc_id}: {msg}")

     



# Insert QC result into qc_log_table
def write_log(qc_id, qc_name,message, status, src_path,tgt_path,job_name,run_id):
    # Escape single quotes for SQL
    message = message.replace("'", "''")
    msg_full = (
        f"Source={src_path}, "
        f"Target={tgt_path}, "
        f"{message}"
    )
    logs_path = dbutils.widgets.get("logs_table_path")

    MAX_RETRIES = 5
    BACKOFF_SECONDS = 10

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            spark.sql(f"""
            INSERT INTO {logs_path}
            (qc_id, qc_name, message, status, cre_ts, job_name, run_id)
            VALUES (
                {qc_id},
                '{qc_name}',
                '{msg_full}',
                '{status}',
                current_timestamp(),
                '{job_name}',
                '{run_id}'
            )
            """)
            # Success → exit loop
            break

        except Exception as e:
            err = str(e)

            if "DELTA_METADATA_CHANGED" in err:
                if attempt == MAX_RETRIES:
                    raise Exception(
                        f"QC log insert failed after {MAX_RETRIES} retries due to Delta metadata conflict"
                    )
                time.sleep(BACKOFF_SECONDS * attempt)  # exponential-ish backoff

    print("Log inserted successfully.")




def insert_qc_error_records_sql(
    qc_id,
    job_name,
    run_id,
    store_error_records,
    error_query,
    err_limit
):
    """
    Inserts QC error records using Spark SQL only
    """
 
    if not store_error_records:
        return
    if not error_query or err_limit is None:
        return
 
    safe_limit = min(int(err_limit), 1000)

    err_path = dbutils.widgets.get("err_table_path")

    spark.sql(f"""
        INSERT INTO {err_path}
        (qc_id, job_name, run_id, err_record, cre_ts)
        SELECT
            {qc_id},
            '{job_name}',
            '{run_id}',
            to_json(struct(*)),
            current_timestamp()
        FROM (
            {error_query}
        ) err_rows
        LIMIT {safe_limit}
    """)

    display_qc_error_records(qc_id,run_id)




def get_api_count(base_api_url):
    """
    Compare API total_records vs. table count.
    Stops job if API count is not greater than table count.
    """

    # 1. Fetch secrets
    client_id = dbutils.secrets.get("ability_api_scope", "CLIENT_ID")
    client_secret = dbutils.secrets.get("ability_api_scope", "CLIENT_SECRET")
    access_key = dbutils.secrets.get("ability_api_scope", "ACCESS_KEY")
    access_secret = dbutils.secrets.get("ability_api_scope", "ACCESS_SECRET")
    account_key = dbutils.secrets.get("ability_api_scope", "ACCOUNT_KEY")

    # 2. Generate Token
    auth_str = f"{client_id}:{client_secret}"
    auth_b64 = base64.b64encode(auth_str.encode("utf-8")).decode("utf-8")

    headers = {
        "Authorization": f"Basic {auth_b64}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    data = {
        "grant_type": "password",
        "username": access_key,
        "password": access_secret,
        "scope": "openid ability:accessapi"
    }

    token_url = "https://idp.myabilitynetwork.com/connect/token"
    response = requests.post(token_url, headers=headers, data=data)

    if response.status_code != 200:
        raise Exception(f"Token generation failed: {response.status_code} {response.text}")

    token = response.json().get("access_token")

      # get widget values (expected format: YYYY-MM-DD)
    start_date = dbutils.widgets.get("start_dt")
    end_date = dbutils.widgets.get("end_dt")

    api_url = f"{base_api_url}?created={start_date},{end_date}"

    # 3. Call API
    headers = {
        "accept": "application/json",
        "X-AccountKey": account_key,
        "Authorization": f"Bearer {token}",
        "X-Request-Mode": "P"
    }

    response = requests.get(api_url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Data fetch failed: {response.status_code} {response.text}")

    data = response.json()

    api_count = data.get("total_records")
    print("API Total Records:", api_count)
    return api_count

    


In [0]:
def display_qc_error_records(qc_id, run_id):

    ERR_TABLE = dbutils.widgets.get("err_table_path")


    # 1 Infer JSON schema dynamically
    schema = spark.sql(f"""
        SELECT schema_of_json(err_record) AS schema
        FROM {ERR_TABLE}
        WHERE qc_id = {qc_id}
        AND run_id = '{run_id}'
        AND err_record IS NOT NULL
        LIMIT 1
    """).collect()[0]["schema"]

    if not schema:
        print("No error records found for this QC run")
        return

    # 2 Fetch all error records
    df = spark.sql(f"""
        SELECT err_record
        FROM {ERR_TABLE}
        WHERE qc_id = {qc_id}
        AND run_id = '{run_id}'
        LIMIT 5
    """)

    # 3 Parse JSON
    clean_df = df.select(
        from_json(col("err_record"), schema).alias("rec")
    )

    # clean_df = final_df.filter(parsed_df)
    print("\n================ Sample Error Records ================")    # 5 Display final table
    display(clean_df.select("rec.*"))


In [0]:
def count_check(
    src_type,
    tgt_type,
    src_query,
    tgt_query,
    threshold,
    expression,
    src_path=None,
    tgt_path=None,
    qc_name="count_check"
):
    """
    Perform count validation between source and target based on their types
    """

    # CASE 1: API → TABLE
    if src_type == "api" and tgt_type == "table":

        if not is_valid_https_url(src_path):
            raise ValueError(f"Invalid or malformed URL: {src_path}")

        src_count = get_api_count(src_path)
        tgt_count = run_query_with_date(tgt_query, qc_name,tgt_path)

        status, msg = compare_count(
            src_count,
            tgt_count,
            src_type,
            tgt_type,
            threshold,
            expression
        )

    # CASE 2: VOLUME ↔ TABLE
    # CASE 3: TABLE ↔ VOLUME
    # CASE 4: TABLE ↔ TABLE
    elif (
        (src_type == "volume" and tgt_type == "table") or
        (src_type == "table" and tgt_type == "volume") or
        (src_type == "table" and tgt_type == "table")
    ):

        src_count = run_query(src_query, qc_name)
        tgt_count = run_query(tgt_query, qc_name)

        status, msg = compare_count(
            src_count,
            tgt_count,
            src_type,
            tgt_type,
            threshold,
            expression
        )

    else:
        raise ValueError(
            f"Unsupported source-target combination: {src_type} → {tgt_type}"
        )

    return status, msg


In [0]:
def compare_count(src_count, target_count,src_type,tgt_type,threshold,expression):

    if (
        (src_type == "api" and tgt_type == "table") or
        (src_type == "volume" and tgt_type == "table") 
    ):
        if src_count > target_count:
                status = "PASSED"
                message = f"Source Count={src_count} > Target Count={target_count} "
        else:
            status = "FAILED"
            message = f"Source Count={src_count} <= Target Count={target_count} "

    elif (
            (src_type == "table" and tgt_type == "volume") or
            (src_type == "table" and tgt_type == "table")
        ):
        if src_count == target_count:
            status = "PASSED"
            message = f"Source Count={src_count} = Target Count={target_count} "
        else:
            status = "FAILED"
            message = f"Source Count={src_count} != Target Count={target_count} "


    else:
        raise Exception(f"Invalid Source Type/Target Type {qc_name}")


    return status, message

In [0]:
def manual_check(
    src_type,
    tgt_type,
    src_query,
    tgt_query,
    threshold,
    expression,
    src_path,
    tgt_path,
    qc_name
):
    """
    Fetch source and target counts, then call manual_check_run
    """

    # Fetch source count
    if src_type in (None, "", "null"):
        src_count = None

    else:  # table / volume
        src_count = run_query(src_query, qc_name)

    # Fetch target count
    if tgt_type in (None, "", "null"):
        tgt_count = None

    else:  # table / volume
        tgt_count = run_query(tgt_query, qc_name)

    # Compare
    return manual_check_run(
        src_count=src_count,
        tgt_count=tgt_count,
        src_type=src_type,
        tgt_type=tgt_type,
        threshold=threshold,
        expression=expression
    )


In [0]:
def manual_check_run(
    src_count,
    tgt_count,
    src_type,
    tgt_type,
    threshold,
    expression
):
    """
    Manual QC comparison between source and target counts
    """
    
    # Validation
    if src_type in (None, "", "null") and tgt_type in (None, "", "null"):
        raise ValueError("Both source and target cannot be null")

    if expression not in ("==", ">=", "<=", "<", ">", "!="):
        raise ValueError(f"Unsupported expression: {expression}")

    if threshold is not None and threshold < 0:
        raise ValueError("threshold must not be negative")
    

    # Decide comparison values
    if threshold is None:
        # Direct comparison: src vs tgt
        left = src_count
        right = tgt_count
        compare_mode = "Direct src vs tgt"
        diff = None
    else:
        # Threshold comparison: diff vs threshold
        if src_type in (None, "", "null"):
            diff = tgt_count
            base_msg = "Source is NULL"
        elif tgt_type in (None, "", "null"):
            diff = src_count
            base_msg = "Target is NULL"
        else:
            diff = abs(src_count - tgt_count)
            base_msg = "Source vs Target"

        left = diff
        right = threshold
        compare_mode = "Diff vs threshold"

    # Expression Evaluation
    if expression == "==":
        status = left == right
    elif expression == ">=":
        status = left >= right
    elif expression == "<=":
        status = left <= right
    elif expression == ">":
        status = left > right
    elif expression == "<":
        status = left < right
    elif expression == "!=":
        status = left != right

    status_str = "PASSED" if status else "FAILED"

    # Message
    msg = (
        f"{compare_mode} | "
        f"src_count={src_count}, "
        f"tgt_count={tgt_count}, "
        f"diff={diff}, "
        f"threshold={threshold}, "
        f"expression='{expression}'"
    )

    return status_str, msg


In [0]:
def columns_check(
    target_columns,
    qc_name,
    threshold,
    src_query=None,
    tgt_query=None
):
    table_query = resolve_single_table_query(
        src_query,
        tgt_query,
        qc_name
    )

    results = {}

    # COLUMN EXIST CHECK
    if qc_name == "column_exist_check":

        if not target_columns:
            raise ValueError("target_columns must be provided for columns_exist_check")
        validate_query_text(table_query, qc_name)
        desc_df = spark.sql(table_query)
        existing_columns = [row.col_name for row in desc_df.collect()]

        for col_name in target_columns:
            if col_name in existing_columns:
                status = "PASSED"
                msg = f"Column '{col_name}' exists"
            else:
                status = "FAILED"
                msg = f"Column '{col_name}' does NOT exist"

            results[col_name] = {
                "status": status,
                "msg": msg
            }

        return results

    else:
        raise ValueError(f"Invalid column QC name: {qc_name}")


In [0]:
def row_hash_duplicate_check(
    src_type=None,
    tgt_type=None,
    src_query=None,
    tgt_query=None,
    threshold=None
):

    # Validate table-only rule
    if src_type == "table":
        active_query = src_query
        side = "SOURCE"

    elif tgt_type == "table":
        active_query = tgt_query
        side = "TARGET"

    else:
        raise ValueError(
            "row_hash_duplicate_check is supported ONLY for table type"
        )

    if not active_query:
        raise ValueError("Table query must be provided for duplicate check")


    # Execute duplicate check
    duplicate_count = run_query(active_query, "row_hash_duplicate_check")


    # Evaluate
    if threshold is None:
        passed = duplicate_count == 0
    else:
        passed = duplicate_count <= threshold

    status = "PASSED" if passed else "FAILED"

    msg = (
        f"{side} TABLE | "
        f"row_hash_duplicate_check | "
        f"duplicate_count={duplicate_count}, "
        f"threshold={threshold}"
    )

    return status, msg


In [0]:
def validate_single_value(df, qc_name):
    row_count = df.count()
    
    if row_count == 0:
        raise Exception(f"[{qc_name}] Query returned ZERO rows, expected exactly ONE value.")

    if row_count > 1:
        raise Exception(f"[{qc_name}] Query returned MULTIPLE rows ({row_count}), expected exactly ONE value.")

    if len(df.columns) != 1:
        raise Exception(
            f"[{qc_name}] Query returned MULTIPLE columns ({len(df.columns)}), expected exactly ONE column."
        )

    value = df.collect()[0][0]

    # Value must be numeric
    if not isinstance(value, (int, float)):
        raise Exception(
            f"[{qc_name}] Query returned NON-NUMERIC value ({value}). "
            f"Expected a numeric aggregation result."
        )

    return value


In [0]:
def validate_query_text(query, qc_name):
    q = query.lower().strip()

    # Rule 1: Query must not contain LIMIT
    if "limit" in q:
        raise Exception(f"[{qc_name}] Invalid QC Query: LIMIT clause is NOT allowed.")

    # Rule 2: Query must contain an aggregate function
    allowed_aggs = ["count(", "sum(", "avg(", "min(", "max(","describe"]
    if not any(agg in q for agg in allowed_aggs):
        raise Exception(
            f"[{qc_name}] Invalid QC Query: Must contain an aggregate "
            "function (COUNT, SUM, MIN, MAX, AVG)."
        )


In [0]:
def is_valid_https_url(url: str) -> bool:
    if not isinstance(url, str) or not url.strip():
        return False

    # Step 1: Parse the URL
    parsed = urlparse(url)

    # Step 2: Must use https
    if parsed.scheme != "https":
        return False

    # Step 3: Must have a valid domain (netloc)
    if not parsed.netloc:
        return False

    # Step 4: Check for malformed characters (basic safety)
    malformed_pattern = re.compile(r"[<>\s\"{}|\\^`]")
    if malformed_pattern.search(url):
        return False

    return True


In [0]:
def resolve_single_table_query(src_query, tgt_query, qc_name):
    """
    Column-level QCs must run on exactly ONE table.
    """

    if src_query and tgt_query:
        raise ValueError(
            f"{qc_name} is a single-table QC. "
            f"Provide ONLY one of src_query or tgt_query."
        )

    if not src_query and not tgt_query:
        raise ValueError(
            f"{qc_name} requires exactly one query "
            f"(src_query OR tgt_query)."
        )

    return src_query or tgt_query
